# Lesson 02 — Your First AWS Batch GPU Job

In Lesson 01 we saw *why* GPUs are faster. Now we'll run our first job **in the cloud** on a real GPU instance.

### What we'll do
1. Submit a job to AWS Batch (it will run on a `g4dn.xlarge` — an NVIDIA T4 GPU)
2. Wait for it to complete
3. Read the output from CloudWatch Logs

No SSH. No instance management. Just submit and wait.

## AWS Batch — 60-second concept map

```
Compute Environment   →  the pool of EC2 instances we can use  (g4dn.xlarge Spot)
      ↓
Job Queue             →  jobs wait here until an instance is free
      ↓
Job Definition        →  template: which Docker image, how much CPU/GPU/RAM
      ↓
Job                   →  a single run, with the env vars you pass at submit time
```

When you submit a job, AWS finds a free instance (or starts a new Spot one), pulls your Docker image from ECR, runs the script, and terminates the instance.

## Step 1 — Check that your .env is loaded

In [ ]:
import os
from dotenv import load_dotenv

# Load from the root .env (two levels up)
load_dotenv(dotenv_path="../../.env")

required = ["AWS_ACCESS_KEY_ID", "BATCH_JOB_QUEUE", "BATCH_JOB_DEFINITION", "ECR_IMAGE_URI"]
missing  = [k for k in required if not os.environ.get(k)]

if missing:
    print(f"❌ Missing in .env: {missing}")
    print("   Please fill in ../../.env before continuing.")
else:
    print("✅ .env loaded — all required keys present")
    print(f"   Job Queue     : {os.environ['BATCH_JOB_QUEUE']}")
    print(f"   Job Definition: {os.environ['BATCH_JOB_DEFINITION']}")

## Step 2 — Look at the job script (runs on GPU inside Batch)

Open `job.py` in this folder. It's ~40 lines:
- Checks that a GPU is present
- Prints GPU name and memory  
- Does a 5000×5000 matrix multiply and times it

This exact file will run inside our Docker container on the cloud GPU.

## Step 3 — Submit the job

In [ ]:
import subprocess, sys

# submit_job.py submits to Batch and polls until done — output streams here
result = subprocess.run(
    [sys.executable, "submit_job.py"],
    capture_output=False,   # let output stream to notebook in real time
)

print("\nExit code:", result.returncode)

## Step 4 — Read the CloudWatch logs

Click the CloudWatch link printed above. You should see:

```
GPU name   : NVIDIA Tesla T4
GPU memory : 15.7 GB
Matrix size: 5000 x 5000
Time       : ~120 ms
Done.
```

**Compare this to Lesson 01**: on your laptop CPU, the same 5000×5000 multiply likely took 3–10 seconds. On the T4 GPU it took ~120 ms — roughly **30–80× faster**.

## Key Takeaway

> **AWS Batch = run a Docker container on a GPU, pay only for what you use.**  
> No servers to manage. No SSH. Perfect for batch ML workloads.

---

## Next lesson → [03 — Video → Frames](../03-video-to-frames/notebook.ipynb)

We'll upload a video to S3 and run a Batch job that extracts frames from it.